<a href="https://colab.research.google.com/github/yeonsub/Learning_Material_for_Creation/blob/main/ExaoneAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P&ID 정보화

기기 인식: YOLO 기반 신경망



문자 인식: OCR로 기기 Tag 추출



연결관계 추출: OpenCV로 배관 선과 접속점을 분석



저장: PostgreSQL은 사용하지 않고 Neo4j 사용

In [ ]:
import torch

from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
import shutil
import zipfile
import os

zip_path = "/content/drive/MyDrive/Colab Notebooks/agent/pid_dataset.zip"
extract_path = "/content/"

# 디렉토리 존재 확인 및 복사/압축 해제 진행
if os.path.exists(zip_path):
    print(f"Copying and extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction completed successfully!")
    # 생성된 경로 확인
    if os.path.exists("/content/pid_dataset"):
        print("Successfully created '/content/pid_dataset'")
        print("Subdirectories:", os.listdir("/content/pid_dataset"))
else:
    print(f"Error: File not found at {zip_path}. Please check your Google Drive path.")

In [ ]:
import json
from pathlib import Path
import yaml
from PIL import Image
import warnings

# 이미지 오픈 시 DecompressionBombError 제한 해제
Image.MAX_IMAGE_PIXELS = None
warnings.simplefilter("ignore", Image.DecompressionBombWarning)

ROOT = Path("/content/pid_dataset")

# 타일링 파라미터 정의
TILE_SIZE = 1024
OVERLAP = 128

def convert_and_tile_split(split):
    image_dir = ROOT / "images" / split
    json_dir = ROOT / "labels_coco_json" / split

    # 원본 JSON 디렉토리가 기존 labels 디렉토리일 경우 백업을 고려하여 처리
    if not json_dir.exists():
        json_dir = ROOT / "labels" / split

    # 새로 생성될 타일링 저장 폴더
    tile_image_dir = ROOT / "images_tiled" / split
    tile_label_dir = ROOT / "labels_tiled" / split

    tile_image_dir.mkdir(parents=True, exist_ok=True)
    tile_label_dir.mkdir(parents=True, exist_ok=True)

    json_files = sorted(json_dir.glob("*.json"))
    if not json_files:
        raise FileNotFoundError(f"No JSON files in {json_dir}")

    categories = {}
    for json_file in json_files:
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        for cat in data["categories"]:
            categories[int(cat["id"])] = cat["name"]

    category_ids = sorted(categories.keys())
    category_map = {cid: idx for idx, cid in enumerate(category_ids)}

    total_tiles = 0
    total_annotations = 0

    for json_file in json_files:
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        image_info = {int(img["id"]): img for img in data["images"]}
        annotations = {}
        for ann in data["annotations"]:
            image_id = int(ann["image_id"])
            annotations.setdefault(image_id, []).append(ann)

        for image_id, img_meta in image_info.items():
            filename = img_meta["file_name"]
            image_path = image_dir / filename

            if not image_path.exists():
                continue

            # 원래 COCO JSON에 정의된 크기 정보
            orig_w = int(img_meta["width"])
            orig_h = int(img_meta["height"])

            with Image.open(image_path) as im:
                current_w, current_h = im.size

                # 이미지 실제 크기와 JSON 메타데이터 비율 조정
                x_ratio = current_w / orig_w
                y_ratio = current_h / orig_h

                y_stride = TILE_SIZE - OVERLAP
                x_stride = TILE_SIZE - OVERLAP

                # 슬라이딩 윈도우 기반 타일 생성
                for y_start in range(0, current_h, y_stride):
                    for x_start in range(0, current_w, x_stride):
                        x_end = min(x_start + TILE_SIZE, current_w)
                        y_end = min(y_start + TILE_SIZE, current_h)

                        tile_w = x_end - x_start
                        tile_h = y_end - y_start

                        # 너무 자투리가 되는 타일은 스킵
                        if tile_w < 128 or tile_h < 128:
                            continue

                        tile_name = f"{Path(filename).stem}_tile_{x_start}_{y_start}.png"
                        tile_img_path = tile_image_dir / tile_name

                        # 1. 이미지 크롭 및 저장
                        tile_img = im.crop((x_start, y_start, x_end, y_end))
                        tile_img.save(tile_img_path)

                        # 2. 바운딩 박스를 크롭된 영역 기준으로 재계산
                        lines = []
                        for ann in annotations.get(image_id, []):
                            cid = int(ann["category_id"])
                            class_idx = category_map[cid]

                            # 원본 기준 COCO 좌표 -> 실제 이미지 비율에 맞춘 절대 좌표 변환
                            ox, oy, ow, oh = map(float, ann["bbox"])
                            ax_min, ay_min = ox * x_ratio, oy * y_ratio
                            ax_max, ay_max = (ox + ow) * x_ratio, (oy + oh) * y_ratio

                            # 타일 로컬 좌표계로 매핑
                            tx_min = max(ax_min - x_start, 0)
                            ty_min = max(ay_min - y_start, 0)
                            tx_max = min(ax_max - x_start, tile_w)
                            ty_max = min(ay_max - y_start, tile_h)

                            tw = tx_max - tx_min
                            th = ty_max - ty_min

                            if tw > 0 and th > 0:
                                original_area = (ax_max - ax_min) * (ay_max - ay_min)
                                intersection_area = tw * th
                                # 객체 면적이 최소 20% 이상 남아있어야 학습 데이터로 편입
                                if (intersection_area / original_area) > 0.2:
                                    xc = (tx_min + tw / 2.0) / tile_w
                                    yc = (ty_min + th / 2.0) / tile_h
                                    wn = tw / tile_w
                                    hn = th / tile_h

                                    lines.append(f"{class_idx} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")

                        tile_lbl_path = tile_label_dir / f"{Path(tile_name).stem}.txt"
                        tile_lbl_path.write_text("\n".join(lines), encoding="utf-8")
                        total_annotations += len(lines)
                        total_tiles += 1

    print(f"[{split}] Completed: Tiles={total_tiles}, Annotations={total_annotations}")
    return categories

# split 별 처리 수행
train_categories = convert_and_tile_split("train")
val_categories = convert_and_tile_split("val")

# 디렉토리 최종 교체 작업
backup_json_path = ROOT / "labels_coco_json"
if ROOT.joinpath("labels").exists() and not backup_json_path.exists():
    ROOT.joinpath("labels").rename(backup_json_path)
    print("Backed up original JSON labels to 'labels_coco_json'")

# 원본 고해상도 이미지 경로 백업
backup_images_path = ROOT / "images_original"
if ROOT.joinpath("images").exists() and not backup_images_path.exists():
    ROOT.joinpath("images").rename(backup_images_path)
    print("Backed up original images to 'images_original'")

# 타일링 폴더를 최종 학습 대상 디렉토리로 배치
if ROOT.joinpath("images_tiled").exists():
    shutil.move(str(ROOT / "images_tiled"), str(ROOT / "images"))
if ROOT.joinpath("labels_tiled").exists():
    shutil.move(str(ROOT / "labels_tiled"), str(ROOT / "labels"))

# data.yaml 생성
category_ids = sorted(train_categories.keys())
names = [train_categories[cid].replace("@", " / ") for cid in category_ids]

yaml_data = {
    "path": str(ROOT),
    "train": "images/train",
    "val": "images/val",
    "nc": len(names),
    "names": names
}

with open(ROOT / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(yaml_data, f, allow_unicode=True, sort_keys=False)

print("Number of classes:", len(names))
print("data.yaml updated with tiled configurations.")

In [ ]:
# 1. 필요한 라이브러리가 없을 경우 자동으로 설치하도록 추가
!pip install -q ultralytics
from ultralytics import YOLO

# 2. 드라이브 가중치를 로컬 환경으로 복사하는 함수 정의
drive_weight_path = Path("/content/drive/MyDrive/Colab Notebooks/agent/best.pt")
local_weight_path = Path("/content/best.pt")

if drive_weight_path.exists():
    print(f"Found weights in Google Drive. Copying to local path: {local_weight_path}")
    shutil.copy(drive_weight_path, local_weight_path)
else:
    print("No model weights found in Google Drive. Fallback to default 'yolo26n.pt'")

# 3. 로컬에 가중치가 존재할 경우 로컬 파일로, 그렇지 않을 경우 default 모델로 초기화
if local_weight_path.exists():
    model_path = str(local_weight_path)
else:
    model_path = "yolo26n.pt"

# 4. 디바이스 체크 및 YOLO 모델 초기화
device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Loading model from: {model_path}")

yoyo_model = YOLO(model_path)

In [ ]:
# CUDA(GPU) 사용 가능 여부 확인 후 디바이스 설정
results = yoyo_model.train(
    data="/content/pid_dataset/data.yaml",
    epochs=1,
    imgsz=1024,
    batch=4,
    patience=20,
    device=device,
    project="/content/pid_training",
    name="pid_model"
)

학습된 신경망 저장

In [ ]:
import shutil
from pathlib import Path

# 가장 최근 학습이 완료된 pid_model-4 하위의 weights/best.pt 지정
source_weight = Path("/content/pid_training/pid_model-4/weights/best.pt")

# 구글 드라이브 대상 저장 경로
target_dir = Path("/content/drive/MyDrive/Colab Notebooks/agent")
target_weight = target_dir / "best.pt"

# 드라이브 폴더가 존재하지 않는 경우 생성 후 복사 진행
if source_weight.exists():
    target_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(source_weight, target_weight)
    print(f"성공적으로 모델 가중치를 구글 드라이브에 저장했습니다:\n{target_weight}")
else:
    print(f"오류: 원본 가중치 파일을 찾을 수 없습니다. 경로를 확인해 주세요: {source_weight}")

검증 이미지에 Box 표시

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from pathlib import Path
import yaml

# 클래스 이름 매핑 로드
try:
    CLASSES = names
except NameError:
    try:
        with open("/content/pid_dataset/data.yaml", "r", encoding="utf-8") as f:
            data_config = yaml.safe_load(f)
            CLASSES = data_config.get("names", [])
    except Exception:
        CLASSES = []

# 이미 로드된 yoyo_model을 검증 모델로 사용
eval_model = yoyo_model
print("Using current session's model in memory (yoyo_model) for evaluation.")

# 타일링 처리된 데이터셋 경로 설정
ROOT = Path("/content/pid_dataset")
train_img_dir = ROOT / "images" / "train"
train_label_dir = ROOT / "labels" / "train"
image_paths = sorted(train_img_dir.glob("*.png"))

# 라벨(GT)이 있는 타일 수집
labeled_image_paths = []
for img_path in image_paths:
    label_path = train_label_dir / f"{img_path.stem}.txt"
    if label_path.exists() and label_path.stat().st_size > 0:
        labeled_image_paths.append(img_path)

print(f"총 {len(image_paths)}개 타일 중 라벨이 존재하는 타일 수: {len(labeled_image_paths)}개")

# 라벨이 있으면서 + 모델 검출(Prediction) 결과가 1개 이상 존재하는 타일을 검색
successful_display_count = 0
max_display = 3

for image_path in labeled_image_paths:
    if successful_display_count >= max_display:
        break

    # 1. 모델 예측 수행
    results = eval_model.predict(source=str(image_path), imgsz=1024, conf=0.1, verbose=False)

    # 예측 상자가 하나도 없으면 다음 타일 탐색
    if len(results[0].boxes) == 0:
        continue

    # 시각화할 이미지 생성
    img = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = img.size

    # 정답 라벨(Ground-Truth) 그리기 (초록색 상자)
    label_path = train_label_dir / f"{image_path.stem}.txt"
    gt_list = []
    if label_path.exists():
        lines = label_path.read_text().splitlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                class_idx = int(parts[0])
                xc, yc, wn, hn = map(float, parts[1:])

                x1 = int((xc - wn/2) * w)
                y1 = int((yc - hn/2) * h)
                x2 = int((xc + wn/2) * w)
                y2 = int((yc + hn/2) * h)

                class_name = CLASSES[class_idx] if class_idx < len(CLASSES) else f"Class_{class_idx}"
                short_name = class_name.split(" / ")[-1]
                gt_list.append(f"- {class_name}")

                draw.rectangle([x1, y1, x2, y2], outline="green", width=3)
                draw.text((x1 + 5, max(0, y1 - 20)), f"[GT] {short_name}", fill="green")

    # 모델 예측값(Prediction) 그리기 (빨간색 상자)
    pred_list = []
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        conf_score = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        class_name = CLASSES[cls_id] if cls_id < len(CLASSES) else f"Class_{cls_id}"
        short_name = class_name.split(" / ")[-1]
        pred_list.append(f"- {class_name} ({conf_score*100:.1f}%)")

        draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
        draw.text((x1 + 5, max(0, y1 - 35)), f"[PRED] {short_name} ({conf_score:.2f})", fill="red")

    # 시각화 출력
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.title(f"Tiled Train Evaluation (With Detections): {image_path.name} (Green: GT, Red: PRED)", fontsize=14, fontweight='bold')
    plt.axis("off")
    plt.show()

    # 정보 출력
    print(f"[{image_path.name}] 상세 기기 목록 비교")
    print("  ● [정답(Ground-Truth)] 실제 기기:")
    if gt_list:
        print("\n".join(f"    {item}" for item in gt_list))
    else:
        print("    (라벨 없음)")

    print("  ● [예측(Prediction)] 모델 탐지:")
    if pred_list:
        print("\n".join(f"    {item}" for item in pred_list))
    else:
        print("    (검출 없음)")
    print("="*80)

    successful_display_count += 1

if successful_display_count == 0:
    print("라벨이 있는 타일 중 현재 모델이 검출(Prediction)에 성공한 타일이 없습니다. 에포크를 늘려 추가 학습이 필요합니다.")

OCR로 Tag 추출
OCR 결과는 FCV-1, PCV-1, P-101 등의 Tag 후보를 추출합니다.

다음으로 OCR 좌표와 YOLO 기기 좌표를 비교해 가까운 기기에 Tag를 연결해야 합니다. 단, Tag가 기호 내부에 있지 않고 옆에 배치되는 경우가 많으므로 단순히 가장 가까운 기기에 붙이는 것은 확정 판정이 아닙니다.

In [ ]:
import easyocr
import re

reader = easyocr.Reader(["en"], gpu=True)

ocr_results = reader.readtext(
    IMAGE_PATH,
    detail=1
)

tag_pattern = re.compile(
    r'\b(?:P|PC|PT|FT|FE|FV|FCV|LCV|HCV|'
    r'XV|V|E|T|F)[- ]?\d+\b',
    re.IGNORECASE
)

tags = []

for bbox, text, confidence in ocr_results:

    text = text.strip()

    if tag_pattern.search(text):

        xs = [p[0] for p in bbox]
        ys = [p[1] for p in bbox]

        tags.append({
            "text": text,
            "center": [
                sum(xs) / len(xs),
                sum(ys) / len(ys)
            ],
            "confidence": float(confidence)
        })

print(tags)

기기 심벌과 OCR Tag 연결
실제로는 개선이 필요합니다
P&ID에서는 Tag가 기호의 위, 아래, 좌우에 배치되므로 중심점 간 거리만으로 판단하면 잘못 연결될 수 있습니다.

따라서 다음 단계에서는 다음 정보를 함께 사용해야 합니다.
- Tag와 기호 사이의 거리
- Tag의 배치 방향
- 기기 종류와 Tag 접두어의 일치 여부 (예: P-101은 Pump)
- 주변 배관 및 계기 연결 구조
- 다른 기기와의 상대적 거리
그리고 한 Tag가 여러 기기에 중복 연결되지 않도록 해야 합니다.

제안: 이 도면을 기준으로 Tag와 기기 사이의 위치 관계를 분석하는 알고리즘을 추가하겠습니다. 그러면 YOLO가 검출한 기기와 OCR이 읽은 Tag를 Neo4j에 함께 저장할 수 있습니다.

In [ ]:
import numpy as np

def get_box_center(bbox):
    x1, y1, x2, y2 = bbox
    return np.array([(x1+x2)/2, (y1+y2)/2])


def match_tags_to_equipment(
    detections,
    tags,
    max_distance=100
):

    matched = []
    unmatched_tags = []

    for tag in tags:

        tag_center = np.array(tag["center"])

        candidates = []

        for i, det in enumerate(detections):

            box_center = get_box_center(det["bbox"])

            distance = np.linalg.norm(
                tag_center - box_center
            )

            if distance < max_distance:
                candidates.append((i, distance))

        candidates.sort(key=lambda x: x[1])

        if candidates:

            idx, distance = candidates[0]

            matched.append({
                "equipment_index": idx,
                "equipment_class": detections[idx]["class"],
                "tag": tag["text"],
                "distance": float(distance),
                "status": "candidate"
            })

        else:

            unmatched_tags.append(tag)

    return matched, unmatched_tags


matched, unmatched = match_tags_to_equipment(
    detections,
    tags
)

print("Matched candidates:")
for item in matched:
    print(item)

print("Unmatched tags:")
for item in unmatched:
    print(item)

배관 선분 검출
우선 OpenCV의 Hough Line Transform으로 직선 후보를 찾겠습니다.이 단계에서는 배관 이외에도 기기 내부의 선, 문자 밑줄, 기호 일부가 검출될 수 있습니다. 따라서 모든 검출 선을 배관이라고 판단하면 안 됩니다.

In [ ]:
def point_to_box_distance(point, bbox):

    x, y = point
    x1, y1, x2, y2 = bbox

    dx = max(x1 - x, 0, x - x2)
    dy = max(y1 - y, 0, y - y2)

    return np.hypot(dx, dy)


connection_candidates = []

for line in line_segments:

    p1 = line["start"]
    p2 = line["end"]

    near1 = []
    near2 = []

    for i, det in enumerate(detections):

        d1 = point_to_box_distance(p1, det["bbox"])
        d2 = point_to_box_distance(p2, det["bbox"])

        if d1 < 15:
            near1.append((i, d1))

        if d2 < 15:
            near2.append((i, d2))

    for i, d1 in near1:

        for j, d2 in near2:

            if i != j:

                connection_candidates.append({
                    "source_index": i,
                    "target_index": j,
                    "line": line,
                    "distance": float(d1 + d2),
                    "status": "candidate"
                })

print("Connection candidates:",
      len(connection_candidates))


기기와 배관 연결 후보 찾기
배관 선분의 양 끝점이 기기 영역에 가까운지 검사하는 간단한 방법입니다.

중요: 이것은 연결관계의 최종 판정 알고리즘이 아닙니다. 배관이 기기 내부까지 이어지는 경우, 여러 선분으로 끊어진 경우, T자 접속, 단순 교차, 신호선 등이 아직 해결되지 않았습니다. 이 코드는 후보를 만들기 위한 첫 단계입니다.

실제 P&ID에서는 선분을 하나씩 추적하여 연속된 배관 경로를 구성하고, 접속점과 단순 교차점을 구분하는 알고리즘을 추가해야 합니다.

In [ ]:
def point_to_box_distance(point, bbox):

    x, y = point
    x1, y1, x2, y2 = bbox

    dx = max(x1 - x, 0, x - x2)
    dy = max(y1 - y, 0, y - y2)

    return np.hypot(dx, dy)


connection_candidates = []

for line in line_segments:

    p1 = line["start"]
    p2 = line["end"]

    near1 = []
    near2 = []

    for i, det in enumerate(detections):

        d1 = point_to_box_distance(p1, det["bbox"])
        d2 = point_to_box_distance(p2, det["bbox"])

        if d1 < 15:
            near1.append((i, d1))

        if d2 < 15:
            near2.append((i, d2))

    for i, d1 in near1:

        for j, d2 in near2:

            if i != j:

                connection_candidates.append({
                    "source_index": i,
                    "target_index": j,
                    "line": line,
                    "distance": float(d1 + d2),
                    "status": "candidate"
                })

print("Connection candidates:",
      len(connection_candidates))

Neo4j 연결 및 저장
이제 데이터베이스 부분입니다.

Neo4j AuraDB를 사용하거나 외부에서 접속 가능한 Neo4j 서버를 준비하면 됩니다. Colab에서 localhost로 접속할 수는 없으므로 Neo4j의 접속 주소가 필요합니다.

Neo4j Python Driver는 공식 라이브러리 neo4j를 사용합니다.

In [ ]:
from neo4j import GraphDatabase
from getpass import getpass

NEO4J_URI = input("Neo4j URI: ")
NEO4J_USER = input("Neo4j Username: ")
NEO4J_PASSWORD = getpass("Neo4j Password: ")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Neo4j connected")

DB 구조 생성
Dawing 도면 정보


 Equipment
  Pump, Vessel 등

   Equipment 연결된 기기
   Instrument 계기 및 밸브

이 코드에서는 아직 Tag가 연결되지 않았으므로 tag=None으로 저장합니다. OCR과 기기 사이의 매칭이 끝나면 Tag를 업데이트하면 됩니다.   

In [ ]:
DRAWING_ID = "PID_TEST_001"

def save_drawing(tx):

    tx.run("""
        MERGE (d:Drawing {id: $id})
        SET d.filename = $filename,
            d.source_type = "image",
            d.created_at = datetime()
    """,
    id=DRAWING_ID,
    filename=IMAGE_PATH
    )


def save_equipment(tx, equipment):

    tx.run("""
        MATCH (d:Drawing {id: $drawing_id})

        MERGE (e:Equipment {id: $id})

        SET e.class = $class,
            e.tag = $tag,
            e.confidence = $confidence,
            e.bbox = $bbox,
            e.extraction_status = "unverified"

        MERGE (e)-[:FOUND_IN]->(d)
    """, **equipment)


with driver.session() as session:

    session.execute_write(save_drawing)

    for i, det in enumerate(detections):

        equipment = {
            "drawing_id": DRAWING_ID,
            "id": f"{DRAWING_ID}_E{i:04d}",
            "class": det["class"],
            "tag": None,
            "confidence": det["confidence"],
            "bbox": det["bbox"]
        }

        session.execute_write(
            save_equipment,
            equipment
        )

print("Equipment saved to Neo4j")

연결 후보 저장

이렇게 하면 Neo4j에서 확정되지 않은 연결을 구분할 수 있습니다. 검증을 통과한 관계만 나중에 CONNECTED_TO로 승인하면 됩니다.

In [ ]:
def save_connection_candidate(tx, source_id,
                              target_id, distance):

    tx.run("""
        MATCH (a:Equipment {id: $source})
        MATCH (b:Equipment {id: $target})

        MERGE (a)-[r:POSSIBLE_CONNECTION
            {drawing_id: $drawing_id}]->(b)

        SET r.distance = $distance,
            r.status = "unverified"
    """,
    source=source_id,
    target=target_id,
    drawing_id=DRAWING_ID,
    distance=distance
    )


with driver.session() as session:

    for c in connection_candidates:

        source_id = (
            f"{DRAWING_ID}_E{c['source_index']:04d}"
        )

        target_id = (
            f"{DRAWING_ID}_E{c['target_index']:04d}"
        )

        session.execute_write(
            save_connection_candidate,
            source_id,
            target_id,
            c["distance"]
        )

print("Connection candidates saved")

Neo4j에서 확인하기
Neo4j Browser에서 다음 Cypher를 실행합니다.

MATCH (e:Equipment)-[:FOUND_IN]->(d:Drawing)
RETURN e, d

MATCH (a:Equipment)-[r:POSSIBLE_CONNECTION]->(b:Equipment)
RETURN a, r, b

MATCH (e:Equipment)
RETURN e.class, count(e) AS count
ORDER BY count DESC

제가 보기에는 신경망 자체보다 학습 데이터 구성과 연결관계 추출이 중요합니다.

현재 도면을 대상으로 다음 세 가지를 먼저 확인하는 것이 좋겠습니다.
1. YOLO가 펌프, 열교환기, 밸브, 계기를 구분할 수 있는가?
2. OCR이 FCV-1, PCV-1, LCV-1 등을 정확히 읽을 수 있는가?
3. 배관이 여러 번 꺾이거나 분기되어도 동일한 연결 경로로 추적할 수 있는가?
특히 세 번째가 중요합니다. 기기 검출은 객체 인식 문제지만, 배관 연결은 도면의 위상 구조(topology)를 복원하는 문제이기 때문입니다.

제가 제안하는 다음 개발 순서는 이렇습니다.
- 먼저 지금 그림을 포함한 P&ID 이미지 10~20장을 모읍니다.
- 기기별 사각형 라벨을 작성합니다.
- YOLO를 학습시킵니다.
- 검출된 기기와 OCR Tag를 연결합니다.
- 배관 연결 알고리즘을 개선합니다.
- 마지막으로 Neo4j에 저장하고 원본 그림 위에 결과를 표시합니다.
처음부터 모든 P&ID 기호를 인식하려고 하지 말고, 우선 펌프·탱크·열교환기·밸브·계기 다섯 종류를 안정적으로 인식하도록 만드는 것이 좋겠습니다.